# Hampton Roads Flood Risk And Climate Impact Report

This notebook is the report runner for the final 7-city regional analysis. It refreshes the database-derived reports, loads the generated CSVs, and displays the tables most useful for the final writeup or presentation.

The core spatial model answers: what roads, buildings, and property-value proxies are exposed if water reaches each modeled elevation? The climate framing added here answers a separate question: how should each water-level scenario be interpreted as a current event, future sea-level-rise planning case, or stress-test possibility?

Frozen regional scope: Norfolk, Virginia Beach, Chesapeake, Hampton, Newport News, Portsmouth, and Suffolk.

## How To Use

Run the notebook top to bottom from the repository root. The heavy spatial workflow should already have been run. This notebook refreshes the final report CSVs from the database and displays them.

If another process is still building GIS data, leave `RUN_FULL_0_TO_6FT_WORKFLOW = False` and `REFRESH_GIS_EXPORTS = False`. The climate-possibility tables use whichever report rows currently exist and will automatically include additional SLR scenarios after the GIS workflow finishes and the report CSVs are refreshed.

If you need to regenerate the city GeoPackages too, set `REFRESH_GIS_EXPORTS = True` in the setup cell.

## 1. Setup

In [14]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

from flood_analysis.db import get_engine

PROJECT_ROOT = Path.cwd()
REPORT_DIR = PROJECT_ROOT / "data" / "processed" / "gis"
ACS_YEAR = 2023
TOP_ROAD_LIMIT = 10
REFRESH_GIS_EXPORTS = False
RUN_FULL_0_TO_6FT_WORKFLOW = False
FUTURE_SLR_DECADES = {
    2030: 0.5,
    2040: 1.0,
    2050: 1.5,
    2060: 2.0,
    2070: 2.5,
    2080: 3.0,
    2090: 3.75,
    2100: 4.5,
}
FULL_SLR_VALUES = ["0", "0.5", "1", "1.5", "2", "2.5", "3", "3.75", "4", "4.5", "5", "6"]
NORFOLK_FULL_SCENARIO_IDS = [
    f"norfolk_pilot_8638610_20260627_plus_{str(float(value)).replace('.', 'p')}ft"
    for value in FULL_SLR_VALUES
]
NORFOLK_SCENARIO_ARGS = [item for scenario_id in NORFOLK_FULL_SCENARIO_IDS for item in ("--scenario-id", scenario_id)]

CITY_EXPORTS = {
    "norfolk_va": "norfolk_flood_exposure_1m.gpkg",
    "virginia_beach_va": "virginia_beach_flood_exposure_coarse.gpkg",
    "chesapeake_va": "chesapeake_flood_exposure_coarse.gpkg",
    "hampton_va": "hampton_flood_exposure_coarse.gpkg",
    "newport_news_va": "newport_news_flood_exposure_coarse.gpkg",
    "portsmouth_va": "portsmouth_flood_exposure_coarse.gpkg",
    "suffolk_va": "suffolk_flood_exposure_coarse.gpkg",
}

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"project_root={PROJECT_ROOT}")
print(f"report_dir={REPORT_DIR}")
print(f"full_slr_values_ft={' '.join(FULL_SLR_VALUES)}")

project_root=/home/rthomson/odu/cs620/project
report_dir=/home/rthomson/odu/cs620/project/data/processed/gis


In [15]:
def run_command(args: list[str]) -> None:
    command = [str(part) for part in args]
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()

def money(value: float) -> str:
    return f"${value:,.0f}"

def load_report(name: str) -> pd.DataFrame:
    path = REPORT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing report: {path}")
    return pd.read_csv(path)

def format_money_columns(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    formatted = frame.copy()
    for column in columns:
        if column in formatted.columns:
            formatted[column] = formatted[column].map(money)
    return formatted

def climate_possibility_band(sea_level_rise_ft: float) -> str:
    value = round(float(sea_level_rise_ft), 2)
    if value == 0:
        return "Observed event benchmark"
    if value <= 1:
        return "Near-term climate possibility"
    if value <= 2:
        return "Mid-century climate possibility"
    if value <= 3:
        return "Late-century high-impact possibility"
    if value <= 4.5:
        return "Long-range high-impact possibility"
    return "Upper stress-test possibility"

def add_climate_possibility(frame: pd.DataFrame) -> pd.DataFrame:
    labeled = frame.copy()
    labeled["climate_possibility"] = labeled["sea_level_rise_ft"].map(climate_possibility_band)
    labeled["probability_note"] = labeled["scenario_type"].map({
        "current_event": "Observed NOAA event baseline, not a recurrence estimate",
        "future_decade": "Scenario-based SLR planning case, not an annual probability",
        "stress_view": "Sensitivity stress view, not a dated forecast",
    }).fillna("Scenario interpretation only")
    return labeled

## 2. Optional Full 0-6 ft Workflow

The established model already supports arbitrary sea-level-rise increments. Set `RUN_FULL_0_TO_6FT_WORKFLOW = True` in the setup cell to rebuild flood views from current conditions through `+6 ft` SLR, including the decade planning curve below. This is heavy because it regenerates rasters, connected extents, road exposure, building exposure, damage, property-value exposure, and report CSVs.

Planning curve: 2030 `+0.5 ft`, 2040 `+1.0 ft`, 2050 `+1.5 ft`, 2060 `+2.0 ft`, 2070 `+2.5 ft`, 2080 `+3.0 ft`, 2090 `+3.75 ft`, 2100 `+4.5 ft`. The `+4`, `+5`, and `+6 ft` outputs are additional stress-view flood layers that account for flooding on top of sea-level rise using the same event baseline.

In [ ]:
if RUN_FULL_0_TO_6FT_WORKFLOW:
    run_command([
        sys.executable,
        "scripts/create_peak_scenarios.py",
        "--event-start",
        "2026-06-27T00:00:00Z",
        "--event-end",
        "2026-06-28T23:59:59Z",
        "--study-area",
        "Norfolk pilot",
        "--sea-level-rise-ft",
        *FULL_SLR_VALUES,
        "--method",
        "1-meter Norfolk screening; connected flood extents; 0-6 ft SLR views",
        "--notes",
        "Flood views through +6 ft, including decade SLR planning curve through 2100.",
    ])
    run_command([sys.executable, "scripts/convert_scenarios_datum.py", "--station", "8638610", "--source-datum", "MLLW", "--target-datum", "NAVD88"])
    run_command([
        sys.executable,
        "scripts/create_flood_depth_rasters.py",
        "--study-area-id",
        "norfolk_va",
        "--dem",
        "data/processed/dem/norfolk_va_usgs_1m_hamptonroads_b23_navd88_m.tif",
        "--dem-units",
        "meters",
        "--output-dir",
        "data/processed/flood_depths_1m",
        *NORFOLK_SCENARIO_ARGS,
    ])
    run_command([sys.executable, "scripts/polygonize_flood_extents.py", "--min-depth-ft", "0", *NORFOLK_SCENARIO_ARGS])
    run_command([sys.executable, "scripts/create_connected_flood_depth_rasters.py", "--output-dir", "data/processed/flood_depths_connected_1m", "--min-depth-ft", "0", *NORFOLK_SCENARIO_ARGS])
    run_command([sys.executable, "scripts/polygonize_connected_flood_extents.py", "--min-depth-ft", "0", *NORFOLK_SCENARIO_ARGS])
    run_command([sys.executable, "scripts/calculate_road_exposure.py", "--study-area-id", "norfolk_va", "--connected"])
    run_command([sys.executable, "scripts/calculate_building_exposure.py", "--study-area-id", "norfolk_va"])
    run_command([sys.executable, "scripts/calculate_building_damage.py", "--study-area-id", "norfolk_va", "--replacement-cost-per-sqft", "175"])
    for study_area_id in [key for key in CITY_EXPORTS if key != "norfolk_va"]:
        run_command([sys.executable, "scripts/run_regional_coarse_workflow.py", "--study-area-id", study_area_id, "--sea-level-rise-ft", *FULL_SLR_VALUES])
else:
    print("Skipping full 0-6 ft rebuild. Set RUN_FULL_0_TO_6FT_WORKFLOW = True to generate expanded flood views.")

## 3. Refresh Report CSVs

These commands update ACS property-value exposure and regenerate all presentation-ready CSV outputs.

In [16]:
run_command([sys.executable, "scripts/calculate_property_value_exposure.py", "--year", str(ACS_YEAR)])
run_command([
    sys.executable,
    "scripts/export_presentation_outputs.py",
    "--output-dir",
    str(REPORT_DIR),
    "--top-road-limit",
    str(TOP_ROAD_LIMIT),
])

$ /home/rthomson/odu/cs620/project/.venv/bin/python scripts/calculate_property_value_exposure.py --year 2023
acs_value study_area_id=chesapeake_va year=2023 median_home_value=388600.0
acs_value study_area_id=hampton_va year=2023 median_home_value=266100.0
acs_value study_area_id=newport_news_va year=2023 median_home_value=283200.0
acs_value study_area_id=norfolk_va year=2023 median_home_value=311200.0
acs_value study_area_id=portsmouth_va year=2023 median_home_value=263900.0
acs_value study_area_id=suffolk_va year=2023 median_home_value=373000.0
acs_value study_area_id=virginia_beach_va year=2023 median_home_value=403200.0
scenario_id=portsmouth_pilot_8638610_20260627_plus_0p0ft study_area_id=portsmouth_va estimated_exposed_property_value=$527,790
scenario_id=portsmouth_pilot_8638610_20260627_plus_1p0ft study_area_id=portsmouth_va estimated_exposed_property_value=$1,746,831
scenario_id=portsmouth_pilot_8638610_20260627_plus_2p0ft study_area_id=portsmouth_va estimated_exposed_property_v

In [17]:
if REFRESH_GIS_EXPORTS:
    for study_area_id, filename in CITY_EXPORTS.items():
        run_command([
            sys.executable,
            "scripts/export_gis_layers.py",
            "--study-area-id",
            study_area_id,
            "--output",
            str(REPORT_DIR / filename),
        ])
else:
    print("Skipping GeoPackage refresh. Set REFRESH_GIS_EXPORTS = True to regenerate GIS exports.")

Skipping GeoPackage refresh. Set REFRESH_GIS_EXPORTS = True to regenerate GIS exports.


## 4. Load Reports

In [18]:
regional = load_report("regional_flood_comparison.csv")
plus_3ft = load_report("regional_chart_plus_3ft_summary.csv")
year_2100 = load_report("regional_chart_2100_summary.csv")
plus_6ft = load_report("regional_chart_plus_6ft_summary.csv")
scenario_lookup = load_report("regional_scenario_lookup.csv")
metric_summary = load_report("regional_metric_summary.csv")
plus_3ft_metrics = load_report("regional_plus_3ft_metric_summary.csv")
top_roads = load_report("regional_top_impacted_roads.csv")
damage_chart = load_report("regional_chart_damage_by_city_scenario.csv")
buildings_chart = load_report("regional_chart_flooded_buildings_by_city_scenario.csv")
roads_chart = load_report("regional_chart_flooded_road_miles_by_city_scenario.csv")
property_chart = load_report("regional_chart_exposed_property_value_by_city_scenario.csv")

print(f"regional_rows={len(regional)}")
print(f"top_impacted_road_rows={len(top_roads)}")

regional_rows=28
top_impacted_road_rows=280


## 5. Climate-Based Flooding Possibilities

The flood rasters and exposure tables are deterministic: they estimate what is exposed if the modeled water surface occurs. They do not calculate annual flood probability. To add climate-based flooding possibilities without overclaiming, this section labels each available SLR scenario as a current-event benchmark, a future planning possibility, or a stress-test possibility.

As the heavier GIS workflow adds more scenarios, these tables will include them automatically after `scripts/export_presentation_outputs.py` refreshes the CSVs.

In [ ]:
scenario_interpretation = add_climate_possibility(scenario_lookup).copy()
scenario_interpretation = scenario_interpretation[[
    "scenario_label",
    "sea_level_rise_ft",
    "planning_year",
    "scenario_type",
    "climate_possibility",
    "probability_note",
]]
scenario_interpretation

In [ ]:
risk_summary = add_climate_possibility(metric_summary).copy()
risk_summary_display = risk_summary[[
    "scenario_label",
    "climate_possibility",
    "city_count",
    "sum_flooded_building_count",
    "sum_flooded_road_miles",
    "sum_estimated_damage_cost",
    "sum_estimated_exposed_property_value",
]].rename(columns={
    "city_count": "cities_with_results",
    "sum_flooded_building_count": "flooded_buildings",
    "sum_flooded_road_miles": "flooded_road_miles",
    "sum_estimated_damage_cost": "estimated_damage_cost",
    "sum_estimated_exposed_property_value": "estimated_exposed_property_value",
})
risk_summary_display["flooded_road_miles"] = risk_summary_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    risk_summary_display[column] = risk_summary_display[column].map(money)
risk_summary_display

In [ ]:
city_climate_risk = add_climate_possibility(regional).copy()
city_climate_risk = city_climate_risk.sort_values([
    "sea_level_rise_ft",
    "estimated_exposed_property_value",
    "estimated_damage_cost",
], ascending=[True, False, False])
top_city_risk = city_climate_risk.groupby("sea_level_rise_ft", as_index=False).head(3)
top_city_risk_display = top_city_risk[[
    "scenario_label",
    "climate_possibility",
    "study_area_name",
    "flooded_building_count",
    "flooded_road_miles",
    "estimated_damage_cost",
    "estimated_exposed_property_value",
]].copy()
top_city_risk_display["flooded_road_miles"] = top_city_risk_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    top_city_risk_display[column] = top_city_risk_display[column].map(money)
top_city_risk_display

## 6. Final +3 ft Results

In [19]:
display_columns = [
    "study_area_name",
    "flooded_building_count",
    "flooded_road_miles",
    "estimated_damage_cost",
    "median_home_value",
    "estimated_exposed_property_value",
]
plus_3ft_display = plus_3ft[display_columns].copy()
plus_3ft_display["flooded_road_miles"] = plus_3ft_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "median_home_value", "estimated_exposed_property_value"]:
    plus_3ft_display[column] = plus_3ft_display[column].map(money)
plus_3ft_display

,study_area_name,flooded_building_count,flooded_road_miles,estimated_damage_cost,median_home_value,estimated_exposed_property_value
0,"Norfolk, VA",1773,43.97,"$205,135,743","$311,200","$354,967,530"
1,"Virginia Beach, VA",589,152.59,"$44,996,669","$403,200","$227,867,202"
2,"Chesapeake, VA",148,41.42,"$23,039,171","$388,600","$55,196,573"
3,"Hampton, VA",241,60.84,"$15,691,151","$266,100","$61,897,682"
4,"Newport News, VA",58,33.06,"$14,763,907","$283,200","$15,136,808"
5,"Portsmouth, VA",20,12.17,"$1,951,872","$263,900","$4,914,142"
6,"Suffolk, VA",11,12.74,"$1,423,761","$373,000","$3,963,915"


## 7. Aggregate Metrics

In [20]:
row = plus_3ft_metrics.iloc[0]
aggregate_rows = [
    {"metric": "Flooded buildings", "total": row.sum_flooded_building_count, "mean": row.mean_flooded_building_count, "median": row.median_flooded_building_count, "min": row.min_flooded_building_count, "max": row.max_flooded_building_count},
    {"metric": "Flooded road count", "total": row.sum_flooded_road_count, "mean": row.mean_flooded_road_count, "median": row.median_flooded_road_count, "min": row.min_flooded_road_count, "max": row.max_flooded_road_count},
    {"metric": "Flooded road miles", "total": row.sum_flooded_road_miles, "mean": row.mean_flooded_road_miles, "median": row.median_flooded_road_miles, "min": row.min_flooded_road_miles, "max": row.max_flooded_road_miles},
    {"metric": "Estimated damage", "total": row.sum_estimated_damage_cost, "mean": row.mean_estimated_damage_cost, "median": row.median_estimated_damage_cost, "min": row.min_estimated_damage_cost, "max": row.max_estimated_damage_cost},
    {"metric": "Exposed property value", "total": row.sum_estimated_exposed_property_value, "mean": row.mean_estimated_exposed_property_value, "median": row.median_estimated_exposed_property_value, "min": row.min_estimated_exposed_property_value, "max": row.max_estimated_exposed_property_value},
]
aggregate = pd.DataFrame(aggregate_rows)
for column in ["total", "mean", "median", "min", "max"]:
    aggregate[column] = aggregate[column].astype(object)
currency_metrics = {"Estimated damage", "Exposed property value"}
for idx, item in aggregate.iterrows():
    if item["metric"] in currency_metrics:
        for column in ["total", "mean", "median", "min", "max"]:
            aggregate.loc[idx, column] = money(float(item[column]))
    else:
        for column in ["total", "mean", "median", "min", "max"]:
            aggregate.loc[idx, column] = round(float(item[column]), 2)
aggregate

,metric,total,mean,median,min,max
0,Flooded buildings,2840.0,405.71,148.0,11.0,1773.0
1,Flooded road count,2479.0,354.14,291.0,99.0,942.0
2,Flooded road miles,356.79,50.97,41.42,12.17,152.59
3,Estimated damage,"$307,002,275","$43,857,468","$15,691,151","$1,423,761","$205,135,743"
4,Exposed property value,"$723,943,853","$103,420,550","$55,196,573","$3,963,915","$354,967,530"


## 8. Scenario Trend Tables

In [21]:
damage_chart

,study_area_name,0.0,1.0,2.0,3.0
0,"Chesapeake, VA",8.022128e+06,1.525365e+07,1.644529e+07,2.303917e+07
1,"Hampton, VA",2.979020e+06,4.550463e+06,7.491594e+06,1.569115e+07
2,"Newport News, VA",1.010652e+07,1.179704e+07,1.271124e+07,1.476391e+07
3,"Norfolk, VA",1.000198e+08,1.019212e+08,1.090638e+08,2.051357e+08
4,"Portsmouth, VA",1.070314e+06,1.127105e+06,1.313575e+06,1.951872e+06
5,"Suffolk, VA",7.756783e+05,8.300043e+05,9.573323e+05,1.423761e+06
6,"Virginia Beach, VA",3.483001e+06,7.660458e+06,2.218911e+07,4.499667e+07


In [22]:
roads_chart

,study_area_name,0.0,1.0,2.0,3.0
0,"Chesapeake, VA",7.578603,12.926326,25.149954,41.416383
1,"Hampton, VA",28.215164,31.930100,42.025077,60.841101
2,"Newport News, VA",24.198259,25.566879,28.210997,33.060444
3,"Norfolk, VA",12.439236,13.723868,19.769958,43.971915
4,"Portsmouth, VA",7.848405,8.559879,9.801033,12.170389
5,"Suffolk, VA",9.451764,10.435147,11.500228,12.738799
6,"Virginia Beach, VA",44.842609,83.428103,114.957218,152.593365


In [23]:
buildings_chart

,study_area_name,0.0,1.0,2.0,3.0
0,"Chesapeake, VA",11,13,30,148
1,"Hampton, VA",11,19,60,241
2,"Newport News, VA",27,28,35,58
3,"Norfolk, VA",169,196,512,1773
4,"Portsmouth, VA",2,7,9,20
5,"Suffolk, VA",7,9,10,11
6,"Virginia Beach, VA",47,134,315,589


In [24]:
property_chart

,study_area_name,0.0,1.0,2.0,3.0
0,"Chesapeake, VA",3.964292e+06,4.998889e+06,1.096254e+07,5.519657e+07
1,"Hampton, VA",2.310419e+06,4.261817e+06,1.466511e+07,6.189768e+07
2,"Newport News, VA",7.242406e+06,7.511395e+06,9.025435e+06,1.513681e+07
3,"Norfolk, VA",4.369496e+07,4.714004e+07,9.621228e+07,3.549675e+08
4,"Portsmouth, VA",5.277895e+05,1.746831e+06,2.274631e+06,4.914142e+06
5,"Suffolk, VA",2.611000e+06,3.245127e+06,3.618127e+06,3.963915e+06
6,"Virginia Beach, VA",1.725426e+07,5.167439e+07,1.220245e+08,2.278672e+08


## 9. Damage Calculations

This section reads feature-level damage estimates directly from PostGIS and rolls them up by city, SLR scenario, damage class, and recovery class. The existing depth-based damage estimate is kept as a screening structural-damage metric. For non-current SLR scenarios, including the `+4`, `+5`, and `+6 ft` flood-on-top-of-SLR stress views, the notebook also reports a total-loss property assumption using ACS median home value times the number of damaged buildings.

In [ ]:
engine = get_engine()
damage_breakdown_query = text(
    """
    SELECT
        d.study_area_id,
        sa.name AS study_area_name,
        d.scenario_id,
        s.sea_level_rise_ft,
        d.damage_class,
        d.recovery_class,
        count(*) AS damaged_building_count,
        sum(d.estimated_structure_value) AS estimated_structure_value,
        sum(d.estimated_damage_cost) AS depth_based_damage_cost,
        avg(d.estimated_damage_cost) AS average_depth_based_damage_cost,
        max(d.max_depth_ft) AS max_depth_ft,
        max(d.estimated_recovery_days) AS max_estimated_recovery_days,
        max(ps.median_home_value) AS median_home_value
    FROM results.connected_building_damage_estimates d
    JOIN processed.study_areas sa
      ON sa.study_area_id = d.study_area_id
    JOIN processed.flood_scenarios s
      ON s.scenario_id = d.scenario_id
    LEFT JOIN results.property_value_exposure_summary ps
      ON ps.study_area_id = d.study_area_id
     AND ps.scenario_id = d.scenario_id
     AND ps.acs_year = :acs_year
    WHERE d.study_area_id = ANY(:study_area_ids)
    GROUP BY d.study_area_id, sa.name, d.scenario_id, s.sea_level_rise_ft, d.damage_class, d.recovery_class
    ORDER BY s.sea_level_rise_ft, sa.name, d.damage_class, d.recovery_class
    """
)
damage_breakdown = pd.read_sql(
    damage_breakdown_query,
    engine,
    params={"acs_year": ACS_YEAR, "study_area_ids": list(CITY_EXPORTS)},
)
slr_to_decade = {v: k for k, v in FUTURE_SLR_DECADES.items()}
damage_breakdown["planning_year"] = damage_breakdown["sea_level_rise_ft"].map(slr_to_decade).astype("Int64")
damage_breakdown["scenario_type"] = damage_breakdown["sea_level_rise_ft"].map(
    lambda value: "current_event" if value == 0 else "future_decade" if value in slr_to_decade else "stress_view"
)
damage_breakdown["scenario_label"] = damage_breakdown.apply(
    lambda row: "Current event"
    if row.sea_level_rise_ft == 0
    else f"{int(row.planning_year)} (+{row.sea_level_rise_ft:g} ft SLR)"
    if pd.notna(row.planning_year)
    else f"+{row.sea_level_rise_ft:g} ft SLR stress view",
    axis=1,
)
damage_breakdown["climate_total_loss_property_value"] = damage_breakdown["damaged_building_count"] * damage_breakdown["median_home_value"].fillna(0)
damage_breakdown.loc[damage_breakdown["sea_level_rise_ft"] == 0, "climate_total_loss_property_value"] = 0
damage_breakdown["loss_assumption"] = damage_breakdown["sea_level_rise_ft"].map(
    lambda value: "depth-based damage only" if value == 0 else "total property loss for SLR/flood stress"
)
print(f"damage_breakdown_rows={len(damage_breakdown)}")

In [ ]:
damage_class_display = damage_breakdown[
    [
        "study_area_name",
        "scenario_label",
        "damage_class",
        "recovery_class",
        "damaged_building_count",
        "depth_based_damage_cost",
        "climate_total_loss_property_value",
        "max_depth_ft",
        "max_estimated_recovery_days",
        "loss_assumption",
    ]
].copy()
damage_class_display["max_depth_ft"] = damage_class_display["max_depth_ft"].round(2)
format_money_columns(damage_class_display, ["depth_based_damage_cost", "climate_total_loss_property_value"])

In [ ]:
damage_loss_summary = damage_breakdown.groupby(
    ["sea_level_rise_ft", "scenario_label", "scenario_type"], as_index=False
).agg(
    damaged_buildings=("damaged_building_count", "sum"),
    depth_based_damage_cost=("depth_based_damage_cost", "sum"),
    climate_total_loss_property_value=("climate_total_loss_property_value", "sum"),
    max_depth_ft=("max_depth_ft", "max"),
    max_estimated_recovery_days=("max_estimated_recovery_days", "max"),
)
damage_loss_summary["max_depth_ft"] = damage_loss_summary["max_depth_ft"].round(2)
format_money_columns(damage_loss_summary, ["depth_based_damage_cost", "climate_total_loss_property_value"])

In [ ]:
stress_damage = damage_breakdown[damage_breakdown["sea_level_rise_ft"] >= 4.0].copy()
if stress_damage.empty:
    print("No +4 ft or higher stress rows available yet. Enable RUN_FULL_0_TO_6FT_WORKFLOW and rerun the notebook to generate them.")
else:
    stress_damage_display = stress_damage.groupby(
        ["study_area_name", "sea_level_rise_ft", "scenario_label"], as_index=False
    ).agg(
        damaged_buildings=("damaged_building_count", "sum"),
        depth_based_damage_cost=("depth_based_damage_cost", "sum"),
        climate_total_loss_property_value=("climate_total_loss_property_value", "sum"),
        max_depth_ft=("max_depth_ft", "max"),
    ).sort_values(["sea_level_rise_ft", "climate_total_loss_property_value"], ascending=[True, False])
    stress_damage_display["max_depth_ft"] = stress_damage_display["max_depth_ft"].round(2)
    display(format_money_columns(stress_damage_display, ["depth_based_damage_cost", "climate_total_loss_property_value"]))

## 10. Top Impacted Roads

In [25]:
top_roads_plus_3ft = top_roads[top_roads["sea_level_rise_ft"] == 3.0].copy()
top_roads_plus_3ft["flooded_length_mi"] = top_roads_plus_3ft["flooded_length_mi"].round(3)
top_roads_plus_3ft[["study_area_name", "rank", "road_name", "mtfcc", "flooded_length_mi", "max_depth_ft"]].head(30)

,study_area_name,rank,road_name,mtfcc,flooded_length_mi,max_depth_ft
30,"Chesapeake, VA",1,State Rte 165,S1200,1.605,6.726005
31,"Chesapeake, VA",2,Mount Pleasant Rd,S1200,1.605,6.726005
32,"Chesapeake, VA",3,Bunch Walnuts Rd,S1400,1.184,6.726005
33,"Chesapeake, VA",4,(unnamed road),S1400,1.083,6.726005
34,"Chesapeake, VA",5,Lake Drummond Cswy,S1400,0.987,6.726005
35,"Chesapeake, VA",6,Indian Creek Rd,S1400,0.877,6.726005
36,"Chesapeake, VA",7,Blackwater Rd,S1400,0.856,6.726005
37,"Chesapeake, VA",8,(unnamed road),S1400,0.850,6.726005
38,"Chesapeake, VA",9,(unnamed road),S1400,0.713,6.726005
39,"Chesapeake, VA",10,State Rte 168,S1200,0.696,6.726005


## 11. Future Decade Results

These tables appear after the optional future SLR workflow has populated the `+4.5 ft` 2100 scenario. They use the same connected-inundation, exposure, damage, and ACS property-value model as the current `+3 ft` reports.

In [ ]:
slr_to_decade = {v: k for k, v in FUTURE_SLR_DECADES.items()}
future_regional = regional[regional["sea_level_rise_ft"].isin(slr_to_decade)].copy()
future_regional["decade"] = future_regional["sea_level_rise_ft"].map(slr_to_decade).astype("Int64")
available_decades = sorted(future_regional["decade"].dropna().unique().tolist())
print(f"available_future_decades={available_decades}")
if 2100 not in available_decades:
    print("2100 (+4.5 ft) is not available yet. Enable RUN_FULL_0_TO_6FT_WORKFLOW and rerun the notebook to generate it.")

In [ ]:
future_decade_summary = future_regional.groupby(["decade", "sea_level_rise_ft"], as_index=False).agg(
    flooded_buildings=("flooded_building_count", "sum"),
    flooded_road_miles=("flooded_road_miles", "sum"),
    estimated_damage_cost=("estimated_damage_cost", "sum"),
    estimated_exposed_property_value=("estimated_exposed_property_value", "sum"),
)
future_decade_display = future_decade_summary.copy()
future_decade_display["flooded_road_miles"] = future_decade_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "estimated_exposed_property_value"]:
    future_decade_display[column] = future_decade_display[column].map(money)
future_decade_display

In [ ]:
future_2100 = future_regional[future_regional["decade"] == 2100].copy()
if future_2100.empty:
    print("No 2100 (+4.5 ft) rows available yet.")
else:
    future_2100 = future_2100.sort_values("estimated_damage_cost", ascending=False)
    future_2100_display = future_2100[display_columns].copy()
    future_2100_display["flooded_road_miles"] = future_2100_display["flooded_road_miles"].round(2)
    for column in ["estimated_damage_cost", "median_home_value", "estimated_exposed_property_value"]:
        future_2100_display[column] = future_2100_display[column].map(money)
    display(future_2100_display)

## 12. +6 ft Stress View

This table shows the upper flood-view layer requested for flooding on top of existing sea-level-rise scenarios. It is a stress view, not a dated projection.

In [ ]:
if plus_6ft.empty:
    print("No +6 ft rows available yet. Enable RUN_FULL_0_TO_6FT_WORKFLOW and rerun the notebook to generate them.")
else:
    plus_6ft_display = plus_6ft.sort_values("estimated_damage_cost", ascending=False)[display_columns].copy()
    plus_6ft_display["flooded_road_miles"] = plus_6ft_display["flooded_road_miles"].round(2)
    for column in ["estimated_damage_cost", "median_home_value", "estimated_exposed_property_value"]:
        plus_6ft_display[column] = plus_6ft_display[column].map(money)
    display(plus_6ft_display)

## 13. Report Files

In [26]:
for path in sorted(REPORT_DIR.glob("regional*.csv")):
    print(path.relative_to(PROJECT_ROOT))

data/processed/gis/regional_chart_damage_by_city_scenario.csv
data/processed/gis/regional_chart_exposed_property_value_by_city_scenario.csv
data/processed/gis/regional_chart_flooded_buildings_by_city_scenario.csv
data/processed/gis/regional_chart_flooded_road_miles_by_city_scenario.csv
data/processed/gis/regional_chart_plus_3ft_summary.csv
data/processed/gis/regional_flood_comparison.csv
data/processed/gis/regional_metric_summary.csv
data/processed/gis/regional_plus_3ft_metric_summary.csv
data/processed/gis/regional_top_impacted_roads.csv


## Key Caveats

- Norfolk uses the high-resolution 1-meter DEM workflow; the other six cities use the coarse regional DEM workflow.
- Results are screening-level static connected-inundation outputs, not a hydrodynamic simulation.
- Climate-possibility bands are scenario interpretation labels, not annual exceedance probabilities or return periods.
- Damage estimates use a simple depth-based replacement-cost model.
- Non-current SLR damage tables add a total-loss property assumption for flooded buildings, including the `+4`, `+5`, and `+6 ft` flood-on-top-of-SLR stress views.
- Property-value exposure uses city-level ACS median owner-occupied home value as a uniform proxy, not parcel assessment values.
- Regional datum conversion currently uses Sewells Point; production work should evaluate spatially varying tidal datums or VDatum.
- Future decade and `+6 ft` stress-view outputs keep the same event baseline and add sea-level-rise increments; they do not model future storm frequency, rainfall, drainage capacity, shoreline adaptation, or development change.